# 调用OpenAI

考虑到API使用速率上限的问题，编写了以下代码；大家需要把这个文件以及utils目录复制到同一path下使用

需要安装openai包

In [ ]:
raw_key_list = []

with open("./raw_keys.txt","r") as f:
    for line in f.readlines():
        key = line.split("----")[2].strip()
        # print(key)
        raw_key_list.append(key)

: 

测试代码，跑出有效结果说明你环境没问题

In [ ]:
import openai
from utils.openai import OpenAIKey,create_response_chat

openai_key = OpenAIKey(raw_key_list)

MODEL = "gpt-3.5-turbo"

response = create_response_chat(
    MODEL,
    prompt_input=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ],
    max_tokens=32,
    temperature=0
)

response

: 

## 批量测试

把所有需要转化的文本存入prompt_list中

In [9]:
from tqdm import tqdm

prompt_list = []
question_list = ["你的姓名是什么？"]
prompt_template = """你是一个酒店前台客服，以下内容是你想要对用户说的问题，请你换一种表述进行提问。你不需要输出其他内容，只需要完成改写。
原始问题：{}
以下给出你的换一种表述的提问："""

for question in question_list:
    prompt_list.append(prompt_template.format(question))


In [10]:
result_list = []

for prompt in tqdm(prompt_list):
    try_times = 0
    while try_times < 5:
        try: 
            result = create_response_chat(
                MODEL,
                prompt_input=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=512,
                temperature=0
            )
            result_list.append(result)
            break
        except Exception as e:
            try_times += 1
            if try_times == 20:
                result_list.append('None')
            if "RateLimitError" in repr(e) or "APIConnectionError" in repr(e) or "AuthenticationError" in repr(e):
                if "per min" in repr(e):
                    pass
                elif "current quota" in repr(e):
                    openai_key.remove_key()
                if openai_key.switch_key() is None:
                    print("All the keys are expired.")
                    exit(0)
            else:
                print("Unknown error.")
                print(repr(e))

100%|██████████| 1/1 [00:03<00:00,  3.67s/it]


In [11]:
result_list

['请问，可以告诉我您的大名吗？']